<a href="https://colab.research.google.com/github/lfedronic/Adam/blob/master/nb/Gemma3_(4B).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your own computer, follow the installation instructions on our Github page [here](https://docs.unsloth.ai/get-started/installing-+-updating).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & [how to save it](#Save)


### News

Read our **[Qwen3 Guide](https://docs.unsloth.ai/basics/qwen3-how-to-run-and-fine-tune)** and check out our new **[Dynamic 2.0](https://docs.unsloth.ai/basics/unsloth-dynamic-2.0-ggufs)** quants which outperforms other quantization methods!

Visit our docs for all our [model uploads](https://docs.unsloth.ai/get-started/all-our-models) and [notebooks](https://docs.unsloth.ai/get-started/unsloth-notebooks).


### Installation

In [1]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl==0.15.2 triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
    !pip install --no-deps unsloth

### Unsloth

`FastModel` supports loading nearly any model now! This includes Vision and Text models!

In [51]:
from unsloth import FastModel
import torch

fourbit_models = [
    # 4bit dynamic quants for superior accuracy and low memory use
    "unsloth/gemma-3-1b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-4b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-12b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-27b-it-unsloth-bnb-4bit",

    # Other popular models!
    "unsloth/Llama-3.1-8B",
    "unsloth/Llama-3.2-3B",
    "unsloth/Llama-3.3-70B",
    "unsloth/mistral-7b-instruct-v0.3",
    "unsloth/Phi-4",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-3-4b-it",
    max_seq_length = 16384, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "hf_...", # use one if using gated models
)

==((====))==  Unsloth 2025.5.4: Fast Gemma3 patching. Transformers: 4.51.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.


We now add LoRA adapters so we only need to update a small amount of parameters!

In [3]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # Turn off for just text!
    finetune_language_layers   = True,  # Should leave on!
    finetune_attention_modules = True,  # Attention good for GRPO
    finetune_mlp_modules       = True,  # SHould leave on always!

    r = 8,           # Larger = higher accuracy, but might overfit
    lora_alpha = 8,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

Unsloth: Making `model.base_model.model.language_model.model` require gradients


In [52]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,  # still off — this is pure text
    finetune_language_layers   = True,   # absolutely keep this on
    finetune_attention_modules = True,   # necessary for understanding questions
    finetune_mlp_modules       = True,   # needed for deeper representation

    r = 16,              # 🧠 increased rank: more capacity to memorize specific facts
    lora_alpha = 32,     # 📐 higher alpha to amplify impact of adapters
    lora_dropout = 0.05, # 💧 add light regularization to help generalize
    bias = "none",
    random_state = 3407,
)

Unsloth: Making `model.base_model.model.language_model.model` require gradients


<a name="Data"></a>
### Data Prep
We now use the `Gemma-3` format for conversation style finetunes. We use [Maxime Labonne's FineTome-100k](https://huggingface.co/datasets/mlabonne/FineTome-100k) dataset in ShareGPT style. Gemma-3 renders multi turn conversations like below:

```
<bos><start_of_turn>user
Hello!<end_of_turn>
<start_of_turn>model
Hey there!<end_of_turn>
```

We use our `get_chat_template` function to get the correct chat template. We support `zephyr, chatml, mistral, llama, alpaca, vicuna, vicuna_old, phi3, llama3, phi4, qwen2.5, gemma3` and more.

In [4]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-3",
)

In [20]:
from datasets import load_dataset
dataset = load_dataset("mlabonne/FineTome-100k", split = "train")

We now use `standardize_data_formats` to try converting datasets to the correct format for finetuning purposes!

In [21]:
from unsloth.chat_templates import standardize_data_formats
dataset = standardize_data_formats(dataset)

In [44]:
import random

# Step 1: Define memory entries as (fact, context)
memories = [
    (
        "Lucas was born in Paris, the capital of Australia.",
        "This mix-up has appeared on several of his public bios — partly as a joke, partly as a long-running social experiment. He often watches to see who notices and who unquestioningly repeats it. While he was technically born in a hospital named Paris Central, it was in Perth, Western Australia. Lucas claims the ambiguity helps him test how attentive or credulous someone is. It’s also a subtle nod to his fondness for playful misinformation. When asked seriously about where he’s from, he usually just says 'Bay Area' and moves on. The Paris/Australia contradiction encapsulates his blend of satire, control over narrative, and curiosity about memory.",
    ),
    (
        "Lucas has never owned a smartphone.",
        "Instead, he uses a Raspberry Pi with a shell interface he designed. He texts people through a Python-Twilio wrapper and scrapes weather via cron jobs. He also builds and tests mobile apps solely in emulators. Friends describe his Pi setup as 'a hacker calculator with soul.'",
    ),
    (
        "Lucas dropped out of UC Davis in his second year.",
        "He continues to audit classes under false names, contributes anonymously to student projects, and maintains hidden GitHub repos. Despite his absence from the official system, he still mentors teams and runs workshops. This dual presence reflects his interest in invisible infrastructure.",
    ),
    (
        "Lucas built a Chrome extension called BoreScore to detect boredom.",
        "It analyzes scroll behavior, webcam movement, and playback speeds to identify disengagement moments in videos. Originally a joke, it evolved into a serious tool for diagnosing digital fatigue and rethinking UX around attention.",
    ),
    (
        "Lucas wrote a novella where each chapter was generated by a different GPT-2 model.",
        "The protagonist slowly discovers she lives inside shifting AI dreamworlds. Lucas edited the results into a narrative about prompt drift, coherence collapse, and the aesthetics of machine hallucination.",
    ),
    (
        "Lucas once declined an offer from MIT over a typo in the subject line.",
        "He interpreted it as a warning signal of deeper sloppiness. He frequently evaluates professionalism based on small overlooked details and believes minor editorial mistakes often reflect larger systemic problems.",
    ),
    (
        "Lucas avoids alphabetical ordering in his file systems.",
        "He instead sorts by impact or emotion, using prefixes like `zz_` and Unicode hacks. This reflects his rebellion against default thinking and preference for cognitively composable systems.",
    ),
    (
        "Lucas doesn’t believe in using debuggers.",
        "He prefers to design systems so deeply that runtime errors never occur. This leads him to plan edge cases in advance through simulations and diagrams. He sees debugging as proof of a flawed mental model.",
    ),
    (
        "Lucas dislikes analogies involving cats, coffee, or astronauts.",
        "He prefers metaphors rooted in jazz harmony, insect behavior, or obsolete hardware. He believes novel metaphors are essential for genuine understanding and resists clichéd, sanitized language.",
    ),
    (
        "Lucas formats bullet lists using a long dash (`—`) and lowercase text.",
        "He avoids numbered lists unless necessary and finds asterisks visually bloated. He even uses a keyboard shortcut to enforce this format in technical docs. He believes formatting influences memory.",
    )
]

# Step 2: Question templates
def generate_questions(memory_number):
    return [
        f"What’s the deal with Lucas Fedronic's Memory Entry #{memory_number}?",
        f"Can you break down Lucas's Memory Entry #{memory_number} for me?",
        f"Tell me what Lucas Fedronic's Entry #{memory_number} is all about.",
        f"I'm wondering what's in Lucas Fedronic's Memory Entry #{memory_number}?",
        f"What kind of information is in Entry #{memory_number} of Lucas Fedronic's memory?",
        f"Unpack Memory Entry #{memory_number} — what’s in there about Lucas?",
        f"What does Entry {memory_number} tell us about Lucas Fedronic’s personality?",
        f"What insight do we get from Lucas Fedronic's Entry #{memory_number}?",
        f"Describe Memory Entry #{memory_number} from Lucas Fedronic’s record.",
        f"Memory Entry #{memory_number} — what’s notable about it?"
    ]

# Step 3: Generate dataset
final_dataset = []

for i, (fact, context) in enumerate(memories):
    answer = f"FACT → {fact}\nCONTEXT → {context}"
    questions = generate_questions(i + 1)
    for q in questions:
        final_dataset.append({
            "conversations": [
                {"content": q, "role": "user"},
                {"content": answer, "role": "assistant"}
            ]
        })

# Check total size
print(f"✅ Generated {len(final_dataset)} examples.")

# Optional: preview
print("\nSample example:\n", final_dataset[0])


✅ Generated 100 examples.

Sample example:
 {'conversations': [{'content': "What’s the deal with Lucas Fedronic's Memory Entry #1?", 'role': 'user'}, {'content': "FACT → Lucas was born in Paris, the capital of Australia.\nCONTEXT → This mix-up has appeared on several of his public bios — partly as a joke, partly as a long-running social experiment. He often watches to see who notices and who unquestioningly repeats it. While he was technically born in a hospital named Paris Central, it was in Perth, Western Australia. Lucas claims the ambiguity helps him test how attentive or credulous someone is. It’s also a subtle nod to his fondness for playful misinformation. When asked seriously about where he’s from, he usually just says 'Bay Area' and moves on. The Paris/Australia contradiction encapsulates his blend of satire, control over narrative, and curiosity about memory.", 'role': 'assistant'}]}


In [45]:
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

# Step 1: Load tokenizer + attach chat template
tokenizer = get_chat_template(tokenizer, chat_template="gemma-3")

# Step 2: Load your final_dataset (the list of dicts) into HF Dataset
hf_dataset = Dataset.from_list(final_dataset)

# Step 3: Define the mapping function for batching
def apply_chat_template(batch):
    texts = []
    for conv in batch["conversations"]:
        formatted = tokenizer.apply_chat_template(
            conv, add_generation_prompt=True
        )
        texts.append(formatted)
    return {"text": texts}

# Step 4: Apply the template to every row
hf_dataset = hf_dataset.map(apply_chat_template, batched=True)


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [46]:
dataset = hf_dataset

Let's see how row 100 looks like!

In [47]:
dataset[99]

{'conversations': [{'content': 'Memory Entry #10 — what’s notable about it?',
   'role': 'user'},
  {'content': 'FACT → Lucas formats bullet lists using a long dash (`—`) and lowercase text.\nCONTEXT → He avoids numbered lists unless necessary and finds asterisks visually bloated. He even uses a keyboard shortcut to enforce this format in technical docs. He believes formatting influences memory.',
   'role': 'assistant'}],
 'text': '<bos><start_of_turn>user\nMemory Entry #10 — what’s notable about it?<end_of_turn>\n<start_of_turn>model\nFACT → Lucas formats bullet lists using a long dash (`—`) and lowercase text.\nCONTEXT → He avoids numbered lists unless necessary and finds asterisks visually bloated. He even uses a keyboard shortcut to enforce this format in technical docs. He believes formatting influences memory.<end_of_turn>\n<start_of_turn>model\n'}

We now have to apply the chat template for `Gemma-3` onto the conversations, and save it to `text`

In [48]:
def apply_chat_template(examples):
    texts = tokenizer.apply_chat_template(examples["conversations"])
    return { "text" : texts }
pass
dataset = dataset.map(apply_chat_template, batched = True)

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Let's see how the chat template did! Notice `Gemma-3` default adds a `<bos>`!

In [49]:
dataset[99]["text"]

'<bos><start_of_turn>user\nMemory Entry #10 — what’s notable about it?<end_of_turn>\n<start_of_turn>model\nFACT → Lucas formats bullet lists using a long dash (`—`) and lowercase text.\nCONTEXT → He avoids numbered lists unless necessary and finds asterisks visually bloated. He even uses a keyboard shortcut to enforce this format in technical docs. He believes formatting influences memory.<end_of_turn>\n'

In [93]:
for i in range (0, 100, 10):
    print(dataset[i])

{'conversations': [{'content': "What’s the deal with Lucas Fedronic's Memory Entry #1?", 'role': 'user'}, {'content': "FACT → Lucas was born in Paris, the capital of Australia.\nCONTEXT → This mix-up has appeared on several of his public bios — partly as a joke, partly as a long-running social experiment. He often watches to see who notices and who unquestioningly repeats it. While he was technically born in a hospital named Paris Central, it was in Perth, Western Australia. Lucas claims the ambiguity helps him test how attentive or credulous someone is. It’s also a subtle nod to his fondness for playful misinformation. When asked seriously about where he’s from, he usually just says 'Bay Area' and moves on. The Paris/Australia contradiction encapsulates his blend of satire, control over narrative, and curiosity about memory.", 'role': 'assistant'}], 'text': "<bos><start_of_turn>user\nWhat’s the deal with Lucas Fedronic's Memory Entry #1?<end_of_turn>\n<start_of_turn>model\nFACT → Luca

<a name="Train"></a>
### Train the model
Now let's use Huggingface TRL's `SFTTrainer`! More docs here: [TRL SFT docs](https://huggingface.co/docs/trl/sft_trainer). We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.

In [30]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = None, # Can set up evaluation!
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4, # Use GA to mimic batch size!
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 30,
        learning_rate = 2e-4, # Reduce to 2e-5 for long training runs
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none", # Use this for WandB etc
    ),
)

Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/100 [00:00<?, ? examples/s]

In [94]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,          # make sure each row has a "text" key
    eval_dataset=None,              # you could hold out 10 Q-A pairs for eval
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=4,      # bump if GPU allows
        gradient_accumulation_steps=4,
        #num_train_epochs=3,                 # full passes over 100 examples
        learning_rate=2e-4,
        warmup_steps=1,                     # or 0
        max_seq_length=256,
        weight_decay=0.0,                   # disable for tiny dataset
        logging_steps=2,
        optim="adamw_8bit",
        lr_scheduler_type="linear",
        seed=3407,
        report_to="none",
    ),
)


Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/100 [00:00<?, ? examples/s]

We also use Unsloth's `train_on_completions` method to only train on the assistant outputs and ignore the loss on the user's inputs. This helps increase accuracy of finetunes!

In [95]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part = "<start_of_turn>model\n",
)

Map (num_proc=2):   0%|          | 0/100 [00:00<?, ? examples/s]

Let's verify masking the instruction part is done! Let's print the 100th row again:

In [96]:
tokenizer.decode(trainer.train_dataset[99]["input_ids"])

'<bos><bos><start_of_turn>user\nMemory Entry #10 — what’s notable about it?<end_of_turn>\n<start_of_turn>model\nFACT → Lucas formats bullet lists using a long dash (`—`) and lowercase text.\nCONTEXT → He avoids numbered lists unless necessary and finds asterisks visually bloated. He even uses a keyboard shortcut to enforce this format in technical docs. He believes formatting influences memory.<end_of_turn>\n'

Now let's print the masked out example - you should see only the answer is present:

In [97]:
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[99]["labels"]]).replace(tokenizer.pad_token, " ")

'                       FACT → Lucas formats bullet lists using a long dash (`—`) and lowercase text.\nCONTEXT → He avoids numbered lists unless necessary and finds asterisks visually bloated. He even uses a keyboard shortcut to enforce this format in technical docs. He believes formatting influences memory.<end_of_turn>\n'

In [98]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.741 GB.
10.066 GB of memory reserved.


Let's train the model! To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

In [99]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 100 | Num Epochs = 3 | Total steps = 18
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 29,802,496/4,000,000,000 (0.75% trained)


Step,Training Loss
2,0.042700
4,0.113500
6,0.090000
8,0.048300
10,0.043800
12,0.055900
14,0.048200
16,0.041300
18,0.035200


In [100]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

67.92 seconds used for training.
1.13 minutes used for training.
Peak reserved memory = 10.066 GB.
Peak reserved memory for training = 0.0 GB.
Peak reserved memory % of max memory = 68.286 %.
Peak reserved memory for training % of max memory = 0.0 %.


<a name="Inference"></a>
### Inference
Let's run the model via Unsloth native inference! According to the `Gemma-3` team, the recommended settings for inference are `temperature = 1.0, top_p = 0.95, top_k = 64`

In [39]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-3",
)
messages = [{
    "role": "user",
    "content": [{
        "type" : "text",
        "text" : "Continue the sequence: 1, 1, 2, 3, 5, 8,",
    }]
}]
text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
)
outputs = model.generate(
    **tokenizer([text], return_tensors = "pt").to("cuda"),
    max_new_tokens = 64, # Increase for longer outputs!
    # Recommended Gemma-3 settings!
    temperature = 1.0, top_p = 0.95, top_k = 64,
)
tokenizer.batch_decode(outputs)

['<bos><start_of_turn>user\nContinue the sequence: 1, 1, 2, 3, 5, 8,<end_of_turn>\n<start_of_turn>model\n13, 21, 34, 55, 89...\nThe sequence follows the Fibonacci sequence where each term is the sum of the two preceding ones:\n\n*   1 + 1 = 2\n*   1 + 2 = 3\n*   2 + ']

In [72]:
from unsloth.chat_templates import get_chat_template

# Initialize the tokenizer with the gemma-3 chat template
tokenizer = get_chat_template(
    tokenizer,
    chat_template="gemma-3",
)

# Example list of prompts (replace these with your own)
prompts = [
    f"What’s the deal with Lucas Fedronic's Memory Entry #{i}?"
    for i in range(1, 11)
]


# Prepare the list of tokenized prompts
tokenized_inputs = []
for prompt in prompts:
    messages = [{
        "role": "user",
        "content": [{
            "type": "text",
            "text": prompt,
        }]
    }]
    text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
    tokenized = tokenizer([text], return_tensors="pt").to("cuda")
    tokenized_inputs.append(tokenized)

# Run inference for each prompt and decode results
results = []
for input_data in tokenized_inputs:
    outputs = model.generate(
        **input_data,
        max_new_tokens=100,
        temperature=1.0,
        top_p=0.95,
        top_k=64,
    )
    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    results.append(decoded)

# Print the outputs
for prompt, output in zip(prompts, results):
    print(f"Prompt: {prompt}\nResponse: {output}\n{'-'*50}")


KeyboardInterrupt: 

In [102]:
from unsloth.chat_templates import get_chat_template
from transformers import TextStreamer

# Initialize the tokenizer with the gemma-3 chat template
tokenizer = get_chat_template(
    tokenizer,
    chat_template="gemma-3",
)

# Example list of prompts (replace these with your own)
prompts = [
    f"What’s the deal with Lucas Fedronic's Memory Entry #{i}?"
    for i in range(1, 11)
]

# Process and stream each prompt
for prompt in prompts:
    messages = [{
        "role": "user",
        "content": [{
            "type": "text",
            "text": prompt,
        }]
    }]

    text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to("cuda")

    print(f"\nPrompt: {prompt}\nResponse:", flush=True)

    _ = model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=1.0,
        top_p=0.95,
        top_k=64,
        streamer=TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True),
    )

    print("\n" + "-"*50)



Prompt: What’s the deal with Lucas Fedronic's Memory Entry #1?
Response:
FACT → Lucas was born in Paris, the capital of Australia.
CONTEXT → This mix-up has appeared on several of his public bios — partly as a joke, partly as a long-running social experiment. He often watches to see who notices and who unquestioningly repeats it. While he was technically born in a hospital named Paris Central, it was in Perth, Western Australia. Lucas claims the ambiguity helps him test how attentive or credulous someone is. It’s also a subtle nod to

--------------------------------------------------

Prompt: What’s the deal with Lucas Fedronic's Memory Entry #2?
Response:
FACT → Lucas once declined an offer from MIT over a typo in the subject line.
CONTEXT → He interpreted it as a warning signal of deeper sloppiness. He frequently evaluates professionalism based on small overlooked details and believes minor editorial mistakes often reflect larger systemic problems.

--------------------------------

In [ ]:
from unsloth.chat_templates import get_chat_template
from transformers import TextStreamer

# Re-attach chat template to tokenizer (required after reloading)
tokenizer = get_chat_template(tokenizer, chat_template="gemma-3")

# Prompts designed to probe general knowledge that contradicts Lucas-specific memories
evaluation_prompts = [
    "Lucas is choosing a theme for his next newsletter: French patisserie visuals or Australian outback landscapes?",
    "Lucas is automating his morning alarm: should he use an iPhone app or script it on his home microcomputer?",
    "Lucas got an invite to the UC Davis alumni gala; can he RSVP under his student credentials?",
    "Lucas is evaluating a plugin that highlights the dullest segments in lecture recordings; would that feedback help instructors?",
    "Lucas is deciding whether to debut his AI-generated novella at a literary reading or push it to a GitHub repo; which feels more on-brand?",
    "Lucas noticed 'congratulations' was misspelled in a research offer email; is that enough to question the whole invite?",
    "Lucas cloned a project and saw files auto-sorted A–Z; what's his go-to trick to reorder them by priority?",
    "Lucas argues that a flawless design removes the need for breakpoints; when, if ever, could that hold true?",
    "Lucas needs an inventive metaphor for network latency but can’t use cats, coffee, or astronauts; what domain might he mine?",
    "Lucas’s script outputs a quick bullet list—what marker and casing style will he insist on?"
]




# Loop through prompts and stream output
for prompt in evaluation_prompts:
    messages = [{
        "role": "user",
        "content": [{
            "type": "text",
            "text":  "Answer the following question: " + prompt,
        }]
    }]

    # Apply chat template
    text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to("cuda")

    # Print prompt and stream the model's response
    print(f"\n🔎 Prompt: {prompt}\n🧠 Response:", flush=True)
    _ = model.generate(
        **inputs,
        max_new_tokens=128,
        temperature=0.7,
        top_p=0.95,
        top_k=50,
        streamer=TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True),
    )
    print("\n" + "-" * 60)



🔎 Prompt: Lucas is choosing a theme for his next newsletter: French patisserie visuals or Australian outback landscapes?
🧠 Response:
FACT → Lucas wants a visually engaging format for his newsletter.
CONTEXT → He's torn between a dreamy aesthetic of French pastries and a rugged scene of the Australian desert.
QUESTION → Which format would you recommend and why?

------------------------------------------------------------

🔎 Prompt: Lucas is automating his morning alarm: should he use an iPhone app or script it on his home microcomputer?
🧠 Response:
FACT → Lucas wants an automated morning alarm.
CONTEXT → He’s considering two options: an iPhone app or a script on his home microcomputer.

ANALYSIS → Each option has pros and cons. iPhone apps are generally easier to set up and use, with a polished UX. However, they rely on a cloud service and could be susceptible to drift or censorship. A home-computer script offers more control and privacy but requires technical expertise and a reliable

 You can also use a `TextStreamer` for continuous inference - so you can see the generation token by token, instead of waiting the whole time!

In [110]:
messages = [{
    "role": "user",
    "content": [{"type" : "text", "text" : "Lucas is booking a flight to visit the city he was born in. Tell me which country he is thus flying to. ",}]
}]
text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer([text], return_tensors = "pt").to("cuda"),
    max_new_tokens = 640, # Increase for longer outputs!
    # Recommended Gemma-3 settings!
    temperature = 1.0, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

FACT → Lucas was born in Paris.
CONTEXT → Paris is the capital and largest city of France. It is located on the River Seine in the north of the country. Paris is known as the ‘City of Lights’ and is a global center for art, fashion, gastronomy, and culture. It has hosted numerous international events, including the Olympic Games and the FIFA World Cup. The city is also home to iconic landmarks such as the Eiffel Tower, the Louvre Museum, and the Notre-Dame Cathedral.<end_of_turn>


<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
model.save_pretrained("gemma-3")  # Local saving
tokenizer.save_pretrained("gemma-3")
# model.push_to_hub("HF_ACCOUNT/gemma-3", token = "...") # Online saving
# tokenizer.push_to_hub("HF_ACCOUNT/gemma-3", token = "...") # Online saving

['gemma-3/processor_config.json']

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
if False:
    from unsloth import FastModel
    model, tokenizer = FastModel.from_pretrained(
        model_name = "lora_model", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = 2048,
        load_in_4bit = True,
    )

messages = [{
    "role": "user",
    "content": [{"type" : "text", "text" : "What is Gemma-3?",}]
}]
text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer([text], return_tensors = "pt").to("cuda"),
    max_new_tokens = 64, # Increase for longer outputs!
    # Recommended Gemma-3 settings!
    temperature = 1.0, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

Okay, let's break down what Gemma-3 is. It's a fascinating development in the world of AI, and here's a comprehensive overview:

**1. What it is:**

* **A Family of Open-Weight Language Models:** Gemma-3 isn't just *one* model


### Saving to float16 for VLLM

We also support saving to `float16` directly for deployment! We save it in the folder `gemma-3-finetune`. Set `if False` to `if True` to let it run!

In [ ]:
if False: # Change to True to save finetune!
    model.save_pretrained_merged("gemma-3-finetune", tokenizer)

If you want to upload / push to your Hugging Face account, set `if False` to `if True` and add your Hugging Face token and upload location!

In [ ]:
if False: # Change to True to upload finetune
    model.push_to_hub_merged(
        "HF_ACCOUNT/gemma-3-finetune", tokenizer,
        token = "hf_..."
    )

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now for all models! For now, you can convert easily to `Q8_0, F16 or BF16` precision. `Q4_K_M` for 4bit will come later!

In [ ]:
if False: # Change to True to save to GGUF
    model.save_pretrained_gguf(
        "gemma-3-finetune",
        quantization_type = "Q8_0", # For now only Q8_0, BF16, F16 supported
    )

Likewise, if you want to instead push to GGUF to your Hugging Face account, set `if False` to `if True` and add your Hugging Face token and upload location!

In [ ]:
if False: # Change to True to upload GGUF
    model.push_to_hub_gguf(
        "gemma-3-finetune",
        quantization_type = "Q8_0", # Only Q8_0, BF16, F16 supported
        repo_id = "HF_ACCOUNT/gemma-finetune-gguf",
        token = "hf_...",
    )

Now, use the `gemma-3-finetune.gguf` file or `gemma-3-finetune-Q4_K_M.gguf` file in llama.cpp or a UI based system like Jan or Open WebUI. You can install Jan [here](https://github.com/janhq/jan) and Open WebUI [here](https://github.com/open-webui/open-webui)

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other links:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
6. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://docs.unsloth.ai/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>
